# Renewable penetration, prices and volatility: descriptive analysis

The panel has hourly regional observations for FY2020–FY2025. All figures here are descriptive associations—not causal estimates.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
panel = pd.read_parquet(root / 'data/processed/nem_region_hour.parquet')
panel.shape, panel[['timestamp', 'region']].agg(['min', 'max'])

((263040, 34),
                     timestamp region
 min 2019-07-01 00:00:00+10:00   NSW1
 max 2025-06-30 23:00:00+10:00   VIC1)

In [2]:
summary = panel.groupby('region', observed=True).agg(
    mean_price=('rrp_aud_mwh', 'mean'),
    negative_price_hour_share=('negative_price_any', 'mean'),
    mean_intrahour_sd=('intrahour_price_sd', 'mean'),
    mean_wind_solar_share=('renewable_share_ws', 'mean'))
summary.round(3)

,mean_price,negative_price_hour_share,mean_intrahour_sd,mean_wind_solar_share
region,,,,
NSW1,107.283,0.084,38.154,0.169
QLD1,103.236,0.143,46.828,0.133
SA1,86.250,0.269,43.729,0.711
TAS1,78.975,0.108,21.601,0.176
VIC1,76.836,0.224,22.728,0.264


In [3]:
panel = panel.assign(share_bin=pd.qcut(panel.renewable_share_ws, 20, duplicates='drop'))
association = panel.groupby(['region', 'share_bin'], observed=True).agg(
    mean_share=('renewable_share_ws', 'mean'),
    mean_price=('rrp_aud_mwh', 'mean'),
    negative_price_hour_share=('negative_price_any', 'mean'))
association.head(15).round(3)

mean_share  mean_price  negative_price_hour_share
region share_bin                                                          
NSW1   (-0.001, 0.0213]       0.014     297.408                      0.011
       (0.0213, 0.0371]       0.030     167.250                      0.005
       (0.0371, 0.0515]       0.045     155.749                      0.005
       (0.0515, 0.066]        0.059     140.425                      0.003
       (0.066, 0.0821]        0.074     127.410                      0.002
       (0.0821, 0.0995]       0.091     109.348                      0.005
       (0.0995, 0.119]        0.109     108.521                      0.003
       (0.119, 0.139]         0.129     111.473                      0.008
       (0.139, 0.162]         0.150      94.253                      0.011
       (0.162, 0.186]         0.173      91.765                      0.023
       (0.186, 0.212]         0.198      93.617                      0.029
       (0.212, 0.243]         0.227      99.879                      0.060
       (0.243, 0.28]          0.261      82.549                      0.089
       (0.28, 0.321]          0.300      75.136                      0.145
       (0.321, 0.37]          0.344      59.740                      0.263

Run `python -m src.descriptive_price_analysis` from the project root to regenerate the four PNG figures and CSV tables under `outputs/`.